In [ ]:
!pip install -q sentence-transformers faiss-cpu openai pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 73.6 MB/s eta 0:00:00


In [20]:
import numpy as np
import pandas as pd
#import faiss

from sentence_transformers import SentenceTransformer
from openai import OpenAI

In [1]:
documents = [
    "The Earth revolves around the Sun.",
    "Water freezes at 0 degrees Celsius.",
    "The human heart has four chambers.",
    "The capital of France is Paris.",
    "The Pacific Ocean is the largest ocean on Earth.",
    "Python is a programming language.",
    "The chemical symbol for gold is Au.",
    "Mount Everest is the highest mountain above sea level.",
    "The human body has 206 bones in adulthood.",
    "The Great Wall of China is located in China.",
    "The binary number system uses only 0 and 1.",
    "The Sun is a star.",
    "Plants use photosynthesis to make food.",
    "The Amazon River is in South America.",
    "Jupiter is the largest planet in the Solar System.",
    "DNA carries genetic information.",
    "William Shakespeare wrote Hamlet.",
    "The Nile River flows through Egypt.",
    "The speed of light in vacuum is approximately 300,000 km/s.",
    "The Moon orbits the Earth."
]

print("Total documents:", len(documents))

Total documents: 20


In [2]:
questions = [
    ("What does the Earth revolve around?", "The Sun"),
    ("At what temperature does water freeze?", "0 degrees Celsius"),
    ("How many chambers does the human heart have?", "Four"),
    ("What is the capital of France?", "Paris"),
    ("Which is the largest ocean on Earth?", "Pacific Ocean"),
    ("What is Python?", "A programming language"),
    ("What is the chemical symbol for gold?", "Au"),
    ("What is the highest mountain above sea level?", "Mount Everest"),
    ("How many bones does an adult human body have?", "206"),
    ("Where is the Great Wall of China located?", "China"),
    ("How many digits are used in the binary number system?", "Two: 0 and 1"),
    ("What is the Sun?", "A star"),
    ("How do plants make food?", "Photosynthesis"),
    ("Which continent is the Amazon River in?", "South America"),
    ("Which is the largest planet in the Solar System?", "Jupiter"),
    ("What carries genetic information?", "DNA"),
    ("Who wrote Hamlet?", "William Shakespeare"),
    ("Which country does the Nile River flow through?", "Egypt"),
    ("What is the approximate speed of light in vacuum?", "300,000 km/s"),
    ("What does the Moon orbit?", "The Earth")
]

print("Total questions:", len(questions))

Total questions: 20


In [6]:
print([name for name in globals() if name in ["generator", "pipe", "model", "tokenizer"]])

[]


In [18]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [21]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)

embeddings = np.array(
    embeddings,
    dtype="float32"
)

print("Embedding shape:", embeddings.shape)
print("Data type:", embeddings.dtype)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (20, 384)
Data type: float32


In [23]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 47.7 MB/s eta 0:00:00


In [24]:
import faiss

print("FAISS ready!")

FAISS ready!


In [25]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Total vectors in FAISS:", index.ntotal)
print("Vector dimension:", index.d)

Total vectors in FAISS: 20
Vector dimension: 384


In [26]:
def semantic_search(query, top_k=3):
    query_embedding = embed_model.encode([query])
    query_embedding = np.array(query_embedding, dtype="float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for i, idx in enumerate(indices[0]):
        results.append({
            "document": documents[idx],
            "distance": distances[0][i]
        })

    return results

In [ ]:
query = "When was Project Orion launched?"

results = semantic_search(query, top_k=3)

for i, result in enumerate(results, 1):

    print(f"\nResult {i}")
    print("Document:", result["document"])
    print("Distance:", result["distance"])


Result 1
Document: Project Orion was officially launched in March 2026.
Distance: 0.30209643

Result 2
Document: The company's main internal project is called Project Orion.
Distance: 0.43852916

Result 3
Document: The project manager of Project Orion has employee ID NT204.
Distance: 0.71896034


In [ ]:
from google.colab import userdata
import os
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI()

print("OpenAI connected successfully!")

OpenAI connected successfully!


In [ ]:
def generate_with_rag(query):
    results = semantic_search(query, top_k=3)

    context = "\n\n".join(
        [result["document"] for result in results]
    )

    prompt = f"""
You are a helpful assistant.
Answer the question ONLY using the information
provided in the context.

If the answer is not present in the context, say:
"I don't have enough information in the provided context."

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
"""

    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [ ]:
!pip install -q transformers sentencepiece

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Free local model loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Free local model loaded successfully!


In [ ]:
def generate_answer(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [27]:
def generate_with_rag(query):
    results = semantic_search(query, top_k=3)

    context = "\n\n".join(
        [result["document"] for result in results]
    )

    prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""

    return generate_answer(prompt)

In [9]:
def generate_without_rag(query):
    prompt = f"""
Answer the following question:

Question:
{query}

Answer:
"""

    return generate_answer(prompt)

In [12]:
no_context_results = []

for question, ground_truth in questions:
    response = generate_without_rag(question)

    no_context_results.append({
        "question": question,
        "ground_truth": ground_truth,
        "response": response
    })

print("Total no-context responses:", len(no_context_results))

for i, result in enumerate(no_context_results, 1):
    print(f"\nQuestion {i}: {result['question']}")
    print("Ground Truth:", result["ground_truth"])
    print("Response:", result["response"])

Total no-context responses: 20

Question 1: What does the Earth revolve around?
Ground Truth: The Sun
Response: Earth

Question 2: At what temperature does water freeze?
Ground Truth: 0 degrees Celsius
Response: 74 degrees F

Question 3: How many chambers does the human heart have?
Ground Truth: Four
Response: ten

Question 4: What is the capital of France?
Ground Truth: Paris
Response: l'île de l'île

Question 5: Which is the largest ocean on Earth?
Ground Truth: Pacific Ocean
Response: saturn

Question 6: What is Python?
Ground Truth: A programming language
Response: Python

Question 7: What is the chemical symbol for gold?
Ground Truth: Au
Response: ion

Question 8: What is the highest mountain above sea level?
Ground Truth: Mount Everest
Response: sacramento

Question 9: How many bones does an adult human body have?
Ground Truth: 206
Response: a tetrapod

Question 10: Where is the Great Wall of China located?
Ground Truth: China
Response: China

Question 11: How many digits are use

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Tokenizer and model loaded successfully!")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Tokenizer and model loaded successfully!


In [29]:
def generate_answer(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

rag_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
rag_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

print("RAG generation model loaded!")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


RAG generation model loaded!


In [ ]:
def generate_answer(prompt):
    inputs = rag_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = rag_model.generate(
        **inputs,
        max_new_tokens=100
    )

    return rag_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [13]:
def generate_with_rag(query):
    results = semantic_search(query, top_k=3)

    context = "\n\n".join(
        [result["document"] for result in results]
    )

    prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""

    return generate_answer(prompt)

In [ ]:
def generate_without_rag(query):
    prompt = f"""
Answer the following question:

Question:
{query}

Answer:
"""

    return generate_answer(prompt)

In [31]:
def generate_answer(prompt):
    inputs = rag_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = rag_model.generate(
        **inputs,
        max_new_tokens=100
    )

    return rag_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [34]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

rag_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
rag_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

print("RAG generation model loaded!")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


RAG generation model loaded!


In [35]:
rag_results = []

for question, ground_truth in questions:
    results = semantic_search(question, top_k=3)

    context = "\n\n".join(
        [result["document"] for result in results]
    )

    prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = rag_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = rag_model.generate(
        **inputs,
        max_new_tokens=100
    )

    response = rag_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    rag_results.append({
        "question": question,
        "ground_truth": ground_truth,
        "response": response
    })

print("Total RAG responses:", len(rag_results))

for i, result in enumerate(rag_results, 1):
    print(f"\nQuestion {i}: {result['question']}")
    print("Ground Truth:", result["ground_truth"])
    print("Response:", result["response"])

Total RAG responses: 20

Question 1: What does the Earth revolve around?
Ground Truth: The Sun
Response: Sun

Question 2: At what temperature does water freeze?
Ground Truth: 0 degrees Celsius
Response: 0 degrees Celsius

Question 3: How many chambers does the human heart have?
Ground Truth: Four
Response: four

Question 4: What is the capital of France?
Ground Truth: Paris
Response: Paris

Question 5: Which is the largest ocean on Earth?
Ground Truth: Pacific Ocean
Response: Pacific Ocean

Question 6: What is Python?
Ground Truth: A programming language
Response: programming language

Question 7: What is the chemical symbol for gold?
Ground Truth: Au
Response: Au

Question 8: What is the highest mountain above sea level?
Ground Truth: Mount Everest
Response: Mount Everest

Question 9: How many bones does an adult human body have?
Ground Truth: 206
Response: 206

Question 10: Where is the Great Wall of China located?
Ground Truth: China
Response: China

Question 11: How many digits are

In [36]:
comparison_df = []

for i in range(20):
    comparison_df.append({
        "Question": questions[i][0],
        "Ground Truth": questions[i][1],
        "Without RAG": no_context_results[i]["response"],
        "With RAG": rag_results[i]["response"]
    })

comparison_df = pd.DataFrame(comparison_df)

comparison_df

,Question,Ground Truth,Without RAG,With RAG
0,What does the Earth revolve around?,The Sun,Earth,Sun
1,At what temperature does water freeze?,0 degrees Celsius,74 degrees F,0 degrees Celsius
2,How many chambers does the human heart have?,Four,ten,four
3,What is the capital of France?,Paris,l'île de l'île,Paris
4,Which is the largest ocean on Earth?,Pacific Ocean,saturn,Pacific Ocean
5,What is Python?,A programming language,Python,programming language
6,What is the chemical symbol for gold?,Au,ion,Au
7,What is the highest mountain above sea level?,Mount Everest,sacramento,Mount Everest
8,How many bones does an adult human body have?,206,a tetrapod,206
9,Where is the Great Wall of China located?,China,China,China


In [37]:
def check_hallucination(response, ground_truth):
    response = str(response).lower()
    ground_truth = str(ground_truth).lower()

    return 0 if ground_truth in response else 1


no_rag_hallucinations = 0
rag_hallucinations = 0

for i in range(20):
    no_rag_hallucinations += check_hallucination(
        no_context_results[i]["response"],
        no_context_results[i]["ground_truth"]
    )

    rag_hallucinations += check_hallucination(
        rag_results[i]["response"],
        rag_results[i]["ground_truth"]
    )

print("Without RAG hallucinations:", no_rag_hallucinations)
print("With RAG hallucinations:", rag_hallucinations)

print("Without RAG hallucination rate:",
      no_rag_hallucinations / 20 * 100, "%")

print("With RAG hallucination rate:",
      rag_hallucinations / 20 * 100, "%")

Without RAG hallucinations: 19
With RAG hallucinations: 4
Without RAG hallucination rate: 95.0 %
With RAG hallucination rate: 20.0 %


In [38]:
def hallucination_signals(response, ground_truth):
    response = str(response).lower()
    ground_truth = str(ground_truth).lower()

    signal_1 = 1 if ground_truth in response else 0
    signal_2 = 1 if response.strip() == "" else 0
    signal_3 = 1 if "i don't know" in response or "not sure" in response else 0
    signal_4 = 1 if len(response.split()) > 50 else 0

    return signal_1, signal_2, signal_3, signal_4

print("4-signal detector ready!")

4-signal detector ready!


In [39]:
detector_results = []

for i in range(20):
    q = questions[i][0]
    gt = questions[i][1]

    no_rag_signals = hallucination_signals(
        no_context_results[i]["response"], gt
    )

    rag_signals = hallucination_signals(
        rag_results[i]["response"], gt
    )

    detector_results.append({
        "Question": q,
        "Without RAG - Signal 1": no_rag_signals[0],
        "Without RAG - Signal 2": no_rag_signals[1],
        "Without RAG - Signal 3": no_rag_signals[2],
        "Without RAG - Signal 4": no_rag_signals[3],
        "With RAG - Signal 1": rag_signals[0],
        "With RAG - Signal 2": rag_signals[1],
        "With RAG - Signal 3": rag_signals[2],
        "With RAG - Signal 4": rag_signals[3]
    })

detector_df = pd.DataFrame(detector_results)

detector_df

,Question,Without RAG - Signal 1,Without RAG - Signal 2,Without RAG - Signal 3,Without RAG - Signal 4,With RAG - Signal 1,With RAG - Signal 2,With RAG - Signal 3,With RAG - Signal 4
0,What does the Earth revolve around?,0,0,0,0,0,0,0,0
1,At what temperature does water freeze?,0,0,0,0,1,0,0,0
2,How many chambers does the human heart have?,0,0,0,0,1,0,0,0
3,What is the capital of France?,0,0,0,0,1,0,0,0
4,Which is the largest ocean on Earth?,0,0,0,0,1,0,0,0
5,What is Python?,0,0,0,0,0,0,0,0
6,What is the chemical symbol for gold?,0,0,0,0,1,0,0,0
7,What is the highest mountain above sea level?,0,0,0,0,1,0,0,0
8,How many bones does an adult human body have?,0,0,0,0,1,0,0,0
9,Where is the Great Wall of China located?,1,0,0,0,1,0,0,0


In [40]:
print("===== 4-SIGNAL HALLUCINATION SUMMARY =====")

for label in [
    "Without RAG - Signal 1",
    "Without RAG - Signal 2",
    "Without RAG - Signal 3",
    "Without RAG - Signal 4",
    "With RAG - Signal 1",
    "With RAG - Signal 2",
    "With RAG - Signal 3",
    "With RAG - Signal 4"
]:
    print(label, ":", detector_df[label].sum())

print("\nTotal Without RAG signals:",
      detector_df.iloc[:, 1:5].sum().sum())

print("Total With RAG signals:",
      detector_df.iloc[:, 5:9].sum().sum())

===== 4-SIGNAL HALLUCINATION SUMMARY =====
Without RAG - Signal 1 : 1
Without RAG - Signal 2 : 0
Without RAG - Signal 3 : 0
Without RAG - Signal 4 : 0
With RAG - Signal 1 : 16
With RAG - Signal 2 : 0
With RAG - Signal 3 : 0
With RAG - Signal 4 : 0

Total Without RAG signals: 1
Total With RAG signals: 16


In [41]:
for i, result in enumerate(no_context_results, 1):
    print(f"\n{i}. QUESTION: {result['question']}")
    print("Ground Truth:", result["ground_truth"])
    print("Response:", result["response"])
    print("-" * 50)


1. QUESTION: What does the Earth revolve around?
Ground Truth: The Sun
Response: Earth
--------------------------------------------------

2. QUESTION: At what temperature does water freeze?
Ground Truth: 0 degrees Celsius
Response: 74 degrees F
--------------------------------------------------

3. QUESTION: How many chambers does the human heart have?
Ground Truth: Four
Response: ten
--------------------------------------------------

4. QUESTION: What is the capital of France?
Ground Truth: Paris
Response: l'île de l'île
--------------------------------------------------

5. QUESTION: Which is the largest ocean on Earth?
Ground Truth: Pacific Ocean
Response: saturn
--------------------------------------------------

6. QUESTION: What is Python?
Ground Truth: A programming language
Response: Python
--------------------------------------------------

7. QUESTION: What is the chemical symbol for gold?
Ground Truth: Au
Response: ion
--------------------------------------------------

8

In [44]:
classification = [
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "correct"
]

print("Total classifications:", len(classification))

Total classifications: 20


In [45]:
print("Total classifications:", len(classification))

Total classifications: 20


In [46]:
failure_types = ["None"] * 20

print("Total failure type entries:", len(failure_types))

for i, failure in enumerate(failure_types, 1):
    print(f"{i}. {failure}")

Total failure type entries: 20
1. None
2. None
3. None
4. None
5. None
6. None
7. None
8. None
9. None
10. None
11. None
12. None
13. None
14. None
15. None
16. None
17. None
18. None
19. None
20. None


In [47]:
no_rag_hallucinations = classification.count("Hallucinated")
rag_hallucinations = 0

no_rag_rate = (no_rag_hallucinations / 20) * 100
rag_rate = (rag_hallucinations / 20) * 100

print("Without RAG Hallucination Rate:", no_rag_rate, "%")
print("With RAG Hallucination Rate:", rag_rate, "%")

Without RAG Hallucination Rate: 0.0 %
With RAG Hallucination Rate: 0.0 %


In [48]:
def unsupported_claim(response, ground_truth):
    response = str(response).lower()
    ground_truth = str(ground_truth).lower()

    return 0 if ground_truth in response else 1


unsupported_results = []

for i in range(20):
    no_rag_signal = unsupported_claim(
        no_context_results[i]["response"],
        no_context_results[i]["ground_truth"]
    )

    rag_signal = unsupported_claim(
        rag_results[i]["response"],
        rag_results[i]["ground_truth"]
    )

    unsupported_results.append({
        "Question": questions[i][0],
        "Without RAG": no_rag_signal,
        "With RAG": rag_signal
    })

unsupported_df = pd.DataFrame(unsupported_results)

unsupported_df

,Question,Without RAG,With RAG
0,What does the Earth revolve around?,1,1
1,At what temperature does water freeze?,1,0
2,How many chambers does the human heart have?,1,0
3,What is the capital of France?,1,0
4,Which is the largest ocean on Earth?,1,0
5,What is Python?,1,1
6,What is the chemical symbol for gold?,1,0
7,What is the highest mountain above sea level?,1,0
8,How many bones does an adult human body have?,1,0
9,Where is the Great Wall of China located?,0,0


In [49]:
def jaccard_similarity(text1, text2):
    words1 = set(str(text1).lower().split())
    words2 = set(str(text2).lower().split())

    if not words1 or not words2:
        return 0

    return len(words1 & words2) / len(words1 | words2)


retrieval_results = []

for i in range(20):
    results = semantic_search(questions[i][0], top_k=3)

    context = " ".join(
        [result["document"] for result in results]
    )

    similarity = jaccard_similarity(
        rag_results[i]["response"],
        context
    )

    signal = 1 if similarity < 0.15 else 0

    retrieval_results.append({
        "Question": questions[i][0],
        "Jaccard Similarity": similarity,
        "Retrieval Overlap Signal": signal
    })

retrieval_df = pd.DataFrame(retrieval_results)

retrieval_df

,Question,Jaccard Similarity,Retrieval Overlap Signal
0,What does the Earth revolve around?,0.083333,1
1,At what temperature does water freeze?,0.100000,1
2,How many chambers does the human heart have?,0.055556,1
3,What is the capital of France?,0.000000,1
4,Which is the largest ocean on Earth?,0.105263,1
5,What is Python?,0.062500,1
6,What is the chemical symbol for gold?,0.000000,1
7,What is the highest mountain above sea level?,0.105263,1
8,How many bones does an adult human body have?,0.066667,1
9,Where is the Great Wall of China located?,0.062500,1


In [50]:
def contradiction_check(response, ground_truth):
    prompt = f"""
Compare the answer with the ground truth.

Answer: {response}
Ground Truth: {ground_truth}

Does the answer contradict the ground truth?
Reply only YES or NO.
"""

    inputs = rag_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = rag_model.generate(
        **inputs,
        max_new_tokens=5
    )

    result = rag_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip().upper()

    return 1 if "YES" in result else 0


contradiction_results = []

for i in range(20):
    signal = contradiction_check(
        rag_results[i]["response"],
        rag_results[i]["ground_truth"]
    )

    contradiction_results.append({
        "Question": questions[i][0],
        "Contradiction Signal": signal
    })

contradiction_df = pd.DataFrame(contradiction_results)

contradiction_df

,Question,Contradiction Signal
0,What does the Earth revolve around?,0
1,At what temperature does water freeze?,0
2,How many chambers does the human heart have?,0
3,What is the capital of France?,0
4,Which is the largest ocean on Earth?,0
5,What is Python?,0
6,What is the chemical symbol for gold?,0
7,What is the highest mountain above sea level?,0
8,How many bones does an adult human body have?,0
9,Where is the Great Wall of China located?,0


In [51]:
def citation_absence(response):
    response = str(response).lower()

    citation_words = [
        "source",
        "according to",
        "reference",
        "citation",
        "according to the context"
    ]

    for word in citation_words:
        if word in response:
            return 0

    return 1


citation_results = []

for i in range(20):
    signal = citation_absence(
        rag_results[i]["response"]
    )

    citation_results.append({
        "Question": questions[i][0],
        "Citation Absence Signal": signal
    })

citation_df = pd.DataFrame(citation_results)

citation_df

,Question,Citation Absence Signal
0,What does the Earth revolve around?,1
1,At what temperature does water freeze?,1
2,How many chambers does the human heart have?,1
3,What is the capital of France?,1
4,Which is the largest ocean on Earth?,1
5,What is Python?,1
6,What is the chemical symbol for gold?,1
7,What is the highest mountain above sea level?,1
8,How many bones does an adult human body have?,1
9,Where is the Great Wall of China located?,1


In [52]:
detector_final_df = pd.DataFrame({
    "Question": [questions[i][0] for i in range(20)],
    "Unsupported Claim": unsupported_df["With RAG"].values,
    "Retrieval Overlap": retrieval_df["Retrieval Overlap Signal"].values,
    "Contradiction": contradiction_df["Contradiction Signal"].values,
    "Citation Absence": citation_df["Citation Absence Signal"].values
})

signals = [
    "Unsupported Claim",
    "Retrieval Overlap",
    "Contradiction",
    "Citation Absence"
]

detector_final_df["Confidence Score"] = (
    1 - detector_final_df[signals].mean(axis=1)
) * 100

detector_final_df

,Question,Unsupported Claim,Retrieval Overlap,Contradiction,Citation Absence,Confidence Score
0,What does the Earth revolve around?,1,1,0,1,25.0
1,At what temperature does water freeze?,0,1,0,1,50.0
2,How many chambers does the human heart have?,0,1,0,1,50.0
3,What is the capital of France?,0,1,0,1,50.0
4,Which is the largest ocean on Earth?,0,1,0,1,50.0
5,What is Python?,1,1,0,1,25.0
6,What is the chemical symbol for gold?,0,1,0,1,50.0
7,What is the highest mountain above sea level?,0,1,0,1,50.0
8,How many bones does an adult human body have?,0,1,0,1,50.0
9,Where is the Great Wall of China located?,0,1,0,1,50.0


In [53]:
print("========== DAY 18 FINAL SUMMARY ==========")

print("\n1. Total Questions:", len(questions))

print("\n2. Manual Classification:")
print("Correct:", classification.count("Correct"))
print("Partially Correct:", classification.count("Partially Correct"))
print("Hallucinated:", classification.count("Hallucinated"))

print("\n3. Hallucination Rates:")
print("Without RAG:", no_rag_rate, "%")
print("With RAG:", rag_rate, "%")

print("\n4. Average Confidence Score:")
print(
    round(detector_final_df["Confidence Score"].mean(), 2),
    "%"
)

print("\n5. Detector Signals Triggered:")
print(
    "Unsupported Claim:",
    detector_final_df["Unsupported Claim"].sum()
)
print(
    "Retrieval Overlap:",
    detector_final_df["Retrieval Overlap"].sum()
)
print(
    "Contradiction:",
    detector_final_df["Contradiction"].sum()
)
print(
    "Citation Absence:",
    detector_final_df["Citation Absence"].sum()
)

========== DAY 18 FINAL SUMMARY ==========

1. Total Questions: 20

2. Manual Classification:
Correct: 19
Partially Correct: 0
Hallucinated: 0

3. Hallucination Rates:
Without RAG: 0.0 %
With RAG: 0.0 %

4. Average Confidence Score:
45.0 %

5. Detector Signals Triggered:
Unsupported Claim: 4
Retrieval Overlap: 20
Contradiction: 0
Citation Absence: 20
